In [ ]:
import sys
import consts
import core
import utils
import pandas as pd
import more_itertools as itools
from typing import Callable
import random

file_path = 'disponibilidad.csv'

In [ ]:
log = utils.setup_logger()

# 1. Ingesta
log.info(f"Cargando datos desde: {file_path}")
raw_df = utils.load_csv(file_path)
df_clean = core.preprocess_dataframe(raw_df)

# 2. Transformación
df_bin = core.get_binary_matrix(df_clean)

# 3. Depuración y Calidad
density, day_f, slot_f = core.get_stats(df_bin)

utils.log_section(log, "Ranking de disponibilidad por ID", density)

restrictive = density[density < consts.MIN_DENSITY_THRESHOLD].index.tolist()
if restrictive:
    log.warning(f"IDs con disponibilidad crítica ( < {consts.MIN_DENSITY_THRESHOLD} slots): {restrictive}")

utils.log_section(log, "Frecuencia de Slots", slot_f.head(10))

# 4. Cálculo
groups, leftovers = core.find_groups_greedy(df_bin, n=consts.DEFAULT_N_GROUPS)

# 5. Salida
utils.log_section(log, "Propuesta de Grupos", pd.DataFrame(groups))

if leftovers:
    log.warning(f"Personas sin asignar: {leftovers}")


In [ ]:
slot_f.index

In [ ]:
df_bin.index

In [ ]:
df_bin.columns

In [ ]:
tot = len(df_bin)
for c in slot_f.index:
    sel = core.get_stats(df_bin)[2].loc[c]
    display(f"Remove {c} -> {sel} | left: {tot - sel}")
    _slot_f = core.get_stats(df_bin[df_bin.loc[:, [c]].any(axis='columns') == 0])
    display(_slot_f[2].head(3))







In [ ]:
core.get_stats(df_bin)[2]

In [ ]:
core.get_stats(df_bin[df_bin.loc[:, ['Wednesday_Morning: 9:00-11:00']].any(axis='columns') == 0])[2].head(3)

In [ ]:
core.get_stats(df_bin[df_bin.loc[:, ['Wednesday_Morning: 9:00-11:00', 'Friday_Midday: 11:10-13:10']].any(axis='columns') == 0])[2]

In [ ]:
def find_groups_greedy(df_bin: pd.DataFrame, n: int) -> tuple[list[dict], list[int]]:
    """Selección iterativa de grupos basada en máxima coincidencia."""
    remaining_ids = list(df_bin.index)
    groups = []

    for i in range(n):
        if not remaining_ids or df_bin.loc[remaining_ids].sum().max() == 0:
            break
        
        current_view = df_bin.loc[remaining_ids]
        best_slot = current_view.sum().idxmax()
        matched_ids = current_view[current_view[best_slot] == 1].index.tolist()
        
        day_info = best_slot.split('_', 1)
        groups.append({
            "Grupo": i + 1,
            "Día": day_info[0],
            "Slot": day_info[1] if len(day_info) > 1 else "",
            "Nº": len(matched_ids),
            "IDs": ";".join(map(str, sorted(matched_ids)))
        })
        
        remaining_ids = [idx for idx in remaining_ids if idx not in matched_ids]
        
    return groups, remaining_ids

def eval_solution(df_bin: pd.DataFrame, solution: list[str], id_selector: int | Callable[list[int], list[int]] | None=None):
    """Evaluación de la solución obtenida. La solución contiene una lista de grupos"""
    remaining_ids = list(df_bin.index)
    groups = []
    if id_selector is None:
        def f_id_selector(l): return l
    elif isinstance(id_selector, int):
        def f_id_selector(l): return l[:id_selector]
    elif hasattr(id_selector, '__call__'):
        def f_id_selector(l): return id_selector(l)
    else:
        raise TypeError("Argument 'id_selector' has the wrong type")

    for i, sel_slot in enumerate(solution):
        if not remaining_ids or df_bin.loc[remaining_ids].sum().max() == 0:
            break
        
        current_view = df_bin.loc[remaining_ids]
        #best_slot = current_view.sum().idxmax()
        #matched_ids = current_view[current_view[best_slot] == 1].index.tolist()
        matched_ids = f_id_selector(current_view[current_view[sel_slot] == 1].index.tolist())
        
        day_info = sel_slot.split('_', 1)
        groups.append({
            "Grupo": i + 1,
            "Día": day_info[0],
            "Slot": day_info[1] if len(day_info) > 1 else "",
            "Nº": len(matched_ids),
            "IDs": ";".join(map(str, sorted(matched_ids)))
        })
        
        remaining_ids = [idx for idx in remaining_ids if idx not in matched_ids]
        
    return groups, remaining_ids


In [ ]:
[list(sset) for sset in itools.powerset(df_bin.columns) if len(sset) == 3]

In [ ]:
def id_selector(l): return random.sample(l, 8) if len(l) > 12 else l
eval_solution(df_bin, ['Tuesday_Afternoon: 14:00-16:00', 'Wednesday_Midday: 11:10-13:10', 'Thursday_Late Afternoon: 16:10-18:10'], id_selector)

In [ ]:
def id_selector(l): return l
eval_solution(df_bin, ['Tuesday_Afternoon: 14:00-16:00', 'Wednesday_Midday: 11:10-13:10', 'Thursday_Late Afternoon: 16:10-18:10'], id_selector)

In [ ]:
df_bin_orig = df_bin.copy()

In [ ]:
%%time
def id_selector(l): return random.sample(l, 12) if len(l) > 12 else l
all_cand_solutions = [list(sset) for sset in itools.powerset(df_bin.columns) if len(sset) == 3]
results = [(cand_sol, eval_solution(df_bin, cand_sol, id_selector)[1]) for cand_sol in all_cand_solutions]
for cand_sol, leftovers in sorted(results, key=lambda s:len(s[1]))[:10]:
    print(f"{cand_sol} -> leftovers: {leftovers} #{len(leftovers)}")

In [ ]:
df_bin[df_bin['Thursday_Late Afternoon: 16:10-18:10'] == 1]

In [ ]:
def check_group(df_bin, group):
    ids = [int(i) for i in group['IDs'].split(';')]
    day_info = f'{group['Día']}_{group['Slot']}'
    if df_bin.loc[ids, day_info].all():
        print(f'Grupo {group['Grupo']} is valid')
    else:
        print(f'Grupo {group['Grupo']} is NOT valid')

In [ ]:
groups_final_selection = [{'Grupo': 1,
   'Día': 'Tuesday',
   'Slot': 'Afternoon: 14:00-16:00',
   'Nº': 10,
   'IDs': '9;10;13;24;25;26;27;32;38;39'},
  {'Grupo': 2,
   'Día': 'Wednesday',
   'Slot': 'Midday: 11:10-13:10',
   'Nº': 10,
   'IDs': '8;14;17;18;19;21;22;23;33;36'},
  {'Grupo': 3,
   'Día': 'Thursday',
   'Slot': 'Late Afternoon: 16:10-18:10',
   'Nº': 9,
   'IDs': '11;12;16;20;28;30;34;35;37'}
]

In [ ]:
for g in groups_final_selection:
    check_group(df_bin, g)

Los slots seleccionados son:
- Martes 14:00-16:00
- Miércoles 11:00-13:00
- Jueves 16:00-18:00